Name: Vedant Shirgaonkar
Roll No.: D114   
SAP ID: 60009230002

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

X_train_text = [
    "Chinese Beijing Chinese",
    "Chinese Chinese Shanghai",
    "Chinese Macao",
    "Tokyo Japan Chinese",
]

X_test_text = ["Chinese Chinese Chinese Tokyo Japan"]

y_train = np.array([1, 1, 1, 0])

print("X_train_text:", X_train_text)
print("X_test_text:", X_test_text)
print("y_train:", y_train)

X_train_text: ['Chinese Beijing Chinese', 'Chinese Chinese Shanghai', 'Chinese Macao', 'Tokyo Japan Chinese']
X_test_text: ['Chinese Chinese Chinese Tokyo Japan']
y_train: [1 1 1 0]


In [ ]:
vectorizer = CountVectorizer(lowercase=True)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [ ]:
V = X_train.shape[1]
classes = np.unique(y_train)
n_classes = len(classes)

In [ ]:
print(f"Vocabulary: {vectorizer.get_feature_names_out()}")
print(f"Vocabulary Size (V): {V}\n")
print(f"X_train (sparse matrix):\n{X_train.toarray()}\n")
print(f"X_test (sparse matrix):\n{X_test.toarray()}\n")
print(f"Classes: {classes}")

Vocabulary: ['beijing' 'chinese' 'japan' 'macao' 'shanghai' 'tokyo']
Vocabulary Size (V): 6

X_train (sparse matrix):
[[1 2 0 0 0 0]
 [0 2 0 0 1 0]
 [0 1 0 1 0 0]
 [0 1 1 0 0 1]]

X_test (sparse matrix):
[[0 3 1 0 0 1]]

Classes: [0 1]


In [ ]:
n_train_docs = X_train.shape[0]
log_prior = np.zeros(n_classes)

In [ ]:
for c_idx, c in enumerate(classes):
    Nc = np.sum(y_train == c)
    prior = Nc / n_train_docs
    log_prior[c_idx] = np.log(prior)

In [ ]:
print("Log Priors (log_prior):")
print(log_prior)

Log Priors (log_prior):
[-1.38629436 -0.28768207]


In [ ]:
alpha = 1
log_lik = np.zeros((n_classes, V))

In [ ]:
for c_idx, c in enumerate(classes):
    X_train_c = X_train[y_train == c]
    word_counts_c = X_train_c.sum(axis=0)
    Tc = word_counts_c.sum()
    likelihoods = (word_counts_c + alpha) / (Tc + alpha * V)
    log_lik[c_idx, :] = np.log(likelihoods)

In [ ]:
print(f"Shape of log-likelihood matrix: {log_lik.shape}")
print("\nLog-Likelihood Matrix (log_lik):")
print(log_lik)

Shape of log-likelihood matrix: (2, 6)

Log-Likelihood Matrix (log_lik):
[[-2.19722458 -1.5040774  -1.5040774  -2.19722458 -2.19722458 -1.5040774 ]
 [-1.94591015 -0.84729786 -2.63905733 -1.94591015 -1.94591015 -2.63905733]]


In [ ]:
def predict(X, log_prior, log_lik):
    scores = X @ log_lik.T + log_prior
    return np.argmax(scores, axis=1)

In [ ]:
y_pred = predict(X_test, log_prior, log_lik)
class_names = ["Not China", "China"]
predicted_class_name = class_names[y_pred[0]]

In [ ]:
print(f"Predicted Class: '{predicted_class_name}' (Label: {y_pred[0]})")

Predicted Class: 'China' (Label: 1)
